# Elevated measured blood pressure — TabPFN · PNS 2013

**FAPESP–Illinois undiagnosed NCD project** — Isabela Venancio da Silva (USP São Paulo) ·
Xuan Lin · Amogh Mannava (UIUC).

One model family, one outcome. Twelve notebooks share this shape, so any two of
them can be compared line for line.

| | |
|---|---|
| **Outcome** | systolic `W00407` >= 140 **or** diastolic `W00408` >= 90 mmHg |
| **Threshold authority** | WHO 2021 and Diretrizes Brasileiras de Hipertensao Arterial 2020, identical at 140/90 |
| **Cohort** | adults who answer *no* to `Q002` (1 yes, 2 pregnancy-only -> undefined, the row drops, 3 no), i.e. never told by a doctor they had this condition |
| **Expected build** | 6,329 rows, prevalence 15.2% |
| **Model** | TabPFN |
| **Dropped as condition-specific** | `dx_hypertension`, `med_hypertension_2w` |

**Why this family.** A tabular foundation model: it is pre-trained and does one forward pass instead of a fit, so there is no hyper-parameter search to run. Task-list line 6 asks for it. Run through the Prior Labs API, which means the design matrix leaves the session; PNS 2013 is public IBGE microdata, so this is acceptable here and would not be for identifiable data.

## What this notebook does not decide

Preprocessing is frozen. Every variable decision comes from
`PNS_preprocessing_registry_v1.xlsx` through `pns_preprocess.py`: 117 declared
variables in, 91 out, 28 dropped, 2 composites, 5 renamed. **To change a
variable, change the workbook, not this notebook.**

Layer 1 (`build_matrix`) runs once and is deterministic. Layer 2
(`build_preprocessor`) — imputation, splines, encoding, scaling — is fitted
inside each cross-validation fold and never sees the test rows.

## Open with the group

`last_bp_measure` (`Q001`, time since blood pressure was last measured) is a predictor again, and `last_glucose_test` returns symmetrically for diabetes. Both are health-service contact items rather than examination results. Read the coefficient with the caveat Amogh raised: people under treatment are often controlled at the time of measurement, so a contact variable can carry an inverted association.

`Q002 = 2`, told only during pregnancy, no longer counts as diagnosed. Those 144 rows leave the base with an undefined gate and are counted in the attrition file, as gestational hypertension is a distinct condition. The undiagnosed cohort is unaffected: they were already outside it.

`n_medications` and `n_chronic` are sums over their blocks and are rebuilt by
`build_matrix` **after** the condition-specific drop. This is the trap that once
produced an AUC of 0.86 with sensitivity 1.000; do not compute either counter
anywhere in this notebook.

## What the group decided, 22/09/2026

Four questions were open in the `notes` sheet. All four are now closed, and each
one is a change to the workbook rather than to any code in this notebook.

1. **`last_bp_measure` and `last_glucose_test` return**, symmetrically:
   `leak_hypertension` and `leak_diabetes` set to 0 for the respective item.
2. **Pregnancy-only diagnoses leave the base.** `Q002 = 2` and `Q030 = 2` make
   the gate undefined instead of counting as diagnosed, and `P005 = 3`, an
   undefined pregnancy, is excluded alongside `P005 = 1`.
3. **Insulin is declared** as `med_diabetes_insulin` (`Q03402`), condition-specific
   for diabetes only.
4. **Cholesterol has no treated definition**, recorded as a limitation in the
   Methods.

Three things also came out of testing the shared build and are fixed in
`pns_modelkit.py`, documented at the point of use: the diagnosis gates were being
imputed, which inflated two cohorts; the one-hot columns and their declared
categories disagreed on type, so Layer 2 could not fit; and three ordinal orders
are declared in the pre-reversal order that `_recode_fixes` already reversed.

`Q031`, `Q061` and `Q06201`–`Q06206` appear in the `outcomes` sheet as
condition-specific, but were never declared in the registry, so there is nothing
to drop.

---
# 1 · Setup

Installs, the data, and the run switches. Nothing here touches the science,
except section 1.3, where the research question is chosen.

## 1.1 · Install

**What this does.** Installs what Colab does not ship for this family: `tabpfn-client`,
plus `openpyxl` for the workbook.

**What to look for.** Nothing, unless it errors.

In [ ]:
!pip install -q tabpfn-client openpyxl

## 1.2 · Data and shared code

**What this does.** Clones the public repository, which holds the survey file,
both dictionaries, the frozen preprocessing (`pns_preprocess.py` and the
registry workbook) and the shared model helper (`pns_modelkit.py`).

**Drive is off by default**, since mounting asks for permission every session.
The run then writes to the session disk and zips itself at the end. Set
`MOUNT_DRIVE = True` to write into the shared folder instead, which also keeps
the built matrix between sessions and saves the minute it takes to rebuild.
Leave the `DRIVE = ...` line in place either way: the configuration cell reads
it whether or not anything is mounted.

**What to look for.** The two sha256 stamps printed by the build in section 2
must match across notebooks. They are what proves twelve runs used one matrix.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/isasaade-23/pns2013-lab-exams-en.git"
REPO     = "/content/pns2013-lab-exams-en"

# Clone, or update a clone this session already has: a session that started
# before the last commit would otherwise keep running the old pns_modelkit.
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "origin", "main"])
    subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"])
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
sys.path.insert(0, os.path.join(REPO, "pipeline"))

print("pipeline at", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"],
                                    capture_output=True, text=True).stdout.strip())

DATA     = os.path.join(REPO, "data", "pns2013_lab_exams.xlsx")
REGISTRY = os.path.join(REPO, "pipeline", "PNS_preprocessing_registry_v1.xlsx")

# Drive is optional and off by default, because mounting asks for permission
# every session. Set MOUNT_DRIVE = True to write straight into the shared
# folder and to keep the built matrix between sessions; left False, the run
# writes to the session disk and zips itself at the end.
#
# DRIVE is defined either way. Do not comment this line out: the configuration
# cell below reads it.
MOUNT_DRIVE = False
DRIVE = "/content/drive/MyDrive/FAPESP_Illinois"

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted:", e)

print("data    ", os.path.exists(DATA))
print("registry", os.path.exists(REGISTRY))

## 1.3 · Configuration

**What this does.** Replaces what used to be command-line flags. Everything
downstream reads these.

| Switch | Meaning |
|---|---|
| `THRESHOLD` | `None` uses the guideline cut. The prespecified sensitivity analysis is `THRESHOLD = (130, 80)` for the ACC/AHA 2017 cut |
| `COHORT` | `"undiagnosed"` is the primary framing (Model A). `"all"` is diagnostic only |
| `N_TRIALS` | tuning budget. 40 finishes inside a free Colab session; `FULL = True` raises it to 150 |
| `REPEATS` | how many times the 5-fold split is repeated inside the search. 2 halves the noise in the objective and doubles the cost |
| `SCORING` | what the search ranks by. `roc_auc` for comparability; `average_precision` weighs the minority class, which is the honest target for diabetes at 3.8% |
| `RS` | 42, fixed, so the split is identical across the twelve notebooks |

**Runtime.** a few minutes, dominated by API round trips.

In [ ]:
import importlib
import numpy as np, pandas as pd
import pns_preprocess as pp
import pns_modelkit as mk

# re-import, in case an older copy was imported earlier in this session
importlib.reload(pp); importlib.reload(mk)

OUTCOME   = "hypertension"
MODEL     = "tabpfn"
THRESHOLD = None          # None = guideline default; see the table above
COHORT    = "undiagnosed"

# outputs: the shared folder when Drive is mounted, the session disk otherwise
BASE   = f"{DRIVE}/02_analysis" if os.path.isdir(DRIVE) else "/content/work"
OUTDIR = os.path.join(BASE, "outputs", "models")
BUILD  = os.path.join(BASE, "outputs", "matrices")
os.makedirs(OUTDIR, exist_ok=True); os.makedirs(BUILD, exist_ok=True)

print("writing to", OUTDIR)

---
# 2 · The frozen matrix

**What this does.** Calls `build_matrix()` from `pns_preprocess.py` — or reuses
a cached build whose data and registry hashes still match — then asserts the
counts reported to the group: **6,329 rows at 15.2%**.

**What to look for.** If the assertion fails, stop. It means the registry or the
survey file moved, and no result from this notebook is comparable to the others
until that is understood.

In [ ]:
bundle = mk.load_or_build(OUTCOME, data=DATA, registry=REGISTRY,
                          outdir=BUILD, threshold=THRESHOLD, cohort=COHORT)
mk.check_frozen(bundle)

X, y = bundle["X"], bundle["y"]
print(f"\n{X.shape[0]} rows x {X.shape[1]} predictors, prevalence {y.mean():.1%}")
print("threshold authority:", bundle["threshold_authority"])
print("data sha", bundle["data_sha"], "| registry sha", bundle["registry_sha"])

a = mk.attrition(bundle, BUILD)
display(a) if a is not None else None

**Participant flow and roles.** The attrition table above is the row accounting
for the flow diagram. Below, where each column enters Layer 2.

In [ ]:
for k, v in bundle["roles"].items():
    print(f"{k:<10} {len(v):>3}  {', '.join(v[:6])}{' ...' if len(v) > 6 else ''}")

blocks = pd.Series({c: bundle["spec"][c]["block"]
                    for c in X.columns if c in bundle["spec"]}).value_counts()
print("\npredictors per block\n", blocks.to_string())

---
# 3 · Split and preprocessor

**What this does.** Stratified 80/20 at `random_state=42`, then builds the Layer 2
`ColumnTransformer`. For TabPFN: scaling **off**,
age spline **off**.

**What to look for.** Train and test prevalence should match to a decimal. The
preprocessor is passed *into* the pipeline, never fitted here.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_score)

Xtr, Xte, ytr, yte = mk.split(bundle)
SPW = mk.pos_weight(ytr)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=mk.RS)
cv_report = cv

def preproc():
    return mk.preprocessor(bundle, spline=False, scale=False)

n_encoded = mk.preprocessor(bundle, spline=False, scale=False,
                            verbose=True).fit_transform(Xtr).shape[1]
print(f"{X.shape[1]} raw predictors -> {n_encoded} encoded columns")
print(f"positives: train {int(ytr.sum())}, test {int(yte.sum())} "
      f"(scale_pos_weight {SPW:.1f})")

---
# 4 · Fit

TabPFN is pre-trained: there is no hyper-parameter search, and a "fit" is the
cost of uploading the training rows. Two configurations are reported — the
default single forward pass and a larger ensemble — which is the TabPFN
equivalent of the default-versus-tuned comparison the other notebooks run.

**Two ways to run it.** `BACKEND = "api"` sends the design matrix to the Prior
Labs API and needs a token. `BACKEND = "local"` runs the open-weights model in
the session, needs no token, and wants a GPU runtime
(Runtime → Change runtime type → T4). The local path is the one to take when the
token is refused.

**The token.** Colab saved keys, named `TABPFN_TOKEN`, generated at
[platform.priorlabs.ai/account/api-keys](https://platform.priorlabs.ai/account/api-keys).
An account password or an old key gives `HTTP 401 Invalid token`. The cell below
checks the token with one cheap authenticated call **before** any training run,
so a bad key fails in two seconds rather than halfway through the fit.

**What leaves the session.** On the API path, the encoded design matrix. PNS 2013
is public IBGE microdata, so that is acceptable here and would not be for
identifiable data. On the local path, nothing leaves.

In [ ]:
import os

BACKEND = "api"            # "api" or "local"
INTERACTIVE_LOGIN = False  # True -> log in through the browser instead of a token

if BACKEND == "api":
    token = os.environ.get("TABPFN_TOKEN", "")
    try:
        from google.colab import userdata
        token = userdata.get("TABPFN_TOKEN") or token
    except Exception as e:
        print("no Colab saved key read:", e)
    token = (token or "").strip()          # a trailing newline is enough to fail
    os.environ["TABPFN_TOKEN"] = token
    print(f"token: {len(token)} characters, ending {token[-4:] if token else '(none)'}")

    import tabpfn_client
    if INTERACTIVE_LOGIN:
        tabpfn_client.interactive_login()
    elif token:
        tabpfn_client.set_access_token(token)

    # one small authenticated call, to fail fast and legibly
    try:
        from tabpfn_client import UserDataClient
        check = UserDataClient.get_data_summary
    except (ImportError, AttributeError) as e:
        check = None
        print("no cheap validation call in this client version:", e)

    try:
        if check:
            check()
            print("token accepted")
    except Exception as e:
        raise SystemExit(
            f"the API refused this token ({type(e).__name__}: {e}).\n"
            "Three ways out:\n"
            "  1. generate a key at platform.priorlabs.ai/account/api-keys and put\n"
            "     it in the Colab saved keys as TABPFN_TOKEN, enabled for this\n"
            "     notebook. An account password is not an API key.\n"
            "  2. set INTERACTIVE_LOGIN = True and re-run this cell to log in\n"
            "     through the browser.\n"
            "  3. set BACKEND = 'local' and re-run: open weights, no token, GPU\n"
            "     runtime recommended.")

    from tabpfn_client import TabPFNClassifier
    print("tabpfn-client ready; model_path='auto' uses the current release")
else:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn"], check=True)
    from tabpfn import TabPFNClassifier
    try:
        import torch
        print("local TabPFN ready; device:",
              "cuda" if torch.cuda.is_available() else "cpu (slow, use a GPU runtime)")
    except Exception:
        print("local TabPFN ready")

## 4.1 · What actually gets sent

**Why this cell exists.** A foundation model that quietly trained on a subsample
would explain a disappointing number, and the claim needs checking rather than
assuming. Three facts, printed before any fit:

1. **The shape sent.** Rows and encoded columns of the training half, which must
   equal the full training set. Nothing in the pipeline samples rows.
2. **The server limits.** The client compares the dataset against them and
   **raises** when they are exceeded, in `validate_train_set`. It does not
   subsample and it does not truncate, so a run that completes used every row
   and every column it was given.
3. **The resolved parameters.** `n_estimators = None` means the checkpoint
   decides, and the current default is 8. That is why passing 8 explicitly
   changed nothing earlier: it was already the default, not an argument being
   dropped. The API also **caps** it at 8, rejecting anything larger with
   `HTTP 422`, so on that backend 8 is both the floor and the ceiling of the
   ensemble. Row subsampling exists only as an opt-in (`inference_config` with
   `SUBSAMPLE_SAMPLES`) and is not set here.

In [ ]:
Xtr_enc = preproc().fit_transform(Xtr)
print(f"training half sent: {Xtr_enc.shape[0]} rows x {Xtr_enc.shape[1]} columns")
print(f"that is {Xtr_enc.shape[0] / len(Xtr):.0%} of the training rows "
      f"and {Xtr_enc.shape[0] * Xtr_enc.shape[1]:,} cells")

if BACKEND == "api":
    try:
        from tabpfn_client.client import ServiceClient
        lim = ServiceClient.get_model_limits()
        m = lim.max_model_limit
        print(f"\nserver limits: {m.train_set_max_rows:,} rows, {m.max_cols:,} columns, "
              f"{m.train_set_max_cells:,} cells")
        print("within limits:",
              Xtr_enc.shape[0] <= m.train_set_max_rows
              and Xtr_enc.shape[1] <= m.max_cols
              and Xtr_enc.shape[0] * Xtr_enc.shape[1] <= m.train_set_max_cells)
    except Exception as e:
        print("could not read the server limits:", e)

probe = TabPFNClassifier()
print("\nparameters as constructed (None = the checkpoint decides):")
for k, v in sorted(probe.get_params().items()):
    if v is not None or k in ("n_estimators", "inference_config"):
        print(f"  {k} = {v!r}")

In [ ]:
from sklearn.pipeline import Pipeline

def make_tabpfn(**kw):
    """One classifier, whichever backend section 4 set up.

    balance_probabilities matters here: the diabetes cohort is 3.8% positive.
    The two backends do not take the same arguments, and client versions differ
    among themselves, so each keyword is dropped rather than allowed to fail."""
    if BACKEND == "local":
        kw.setdefault("device", "auto")
    else:
        kw.setdefault("model_path", "auto")
    for attempt in ({"balance_probabilities": True, "random_state": mk.RS, **kw},
                    {"random_state": mk.RS, **kw},
                    kw):
        try:
            return TabPFNClassifier(**attempt)
        except TypeError:
            continue
    return TabPFNClassifier()

pipe_def = Pipeline([("prep", preproc()), ("clf", make_tabpfn())])
pipe_def.fit(Xtr, ytr)
row_def, p_def = mk.evaluate(pipe_def, Xte, yte, "TabPFN (default)")
print(f"default  test AUC {row_def['AUC_test']:.3f} "
      f"[{row_def['AUC_lo']:.3f}, {row_def['AUC_hi']:.3f}]")

# The second configuration, and which one it can be depends on the backend.
# The API caps n_estimators at 8 and 8 is also the default, so there is no
# larger ensemble to ask for: the only contrast left is downward, a single
# forward pass against the ensemble of eight, which says what the ensembling
# is worth. Locally there is no cap, so 32 is used.
ALT = 32 if BACKEND == "local" else 1
ALT_LABEL = f"ensemble {ALT}" if ALT > 1 else "single pass"

pipe_alt = Pipeline([("prep", preproc()), ("clf", make_tabpfn(n_estimators=ALT))])
pipe_alt.fit(Xtr, ytr)
row_alt, p_alt = mk.evaluate(pipe_alt, Xte, yte, f"TabPFN ({ALT_LABEL})")
print(f"{ALT_LABEL:<13} test AUC {row_alt['AUC_test']:.3f} "
      f"[{row_alt['AUC_lo']:.3f}, {row_alt['AUC_hi']:.3f}]")

# The bigger ensemble is the base for the variants. When the alternative is the
# single pass, that is the default of 8, chosen on the configuration rather than
# on the test score, which would be selection on the outcome.
if ALT > 1:
    pipe_tuned, row_tuned, p_tuned, n_best = pipe_alt, row_alt, p_alt, ALT
else:
    pipe_tuned, row_tuned, p_tuned, n_best = pipe_def, row_def, p_def, 8

BEST_MODEL, BEST_P, BEST_PARAMS = pipe_tuned, p_tuned, {"n_estimators": n_best,
                                                        "backend": BACKEND}

**A 5-fold CV score, for comparability with the other three notebooks.** Five
more round trips; skip it if the API budget is tight.

In [ ]:
RUN_CV = True

if RUN_CV:
    from sklearn.model_selection import cross_val_score
    cv_scores = cross_val_score(Pipeline([("prep", preproc()), ("clf", make_tabpfn())]),
                                Xtr, ytr, cv=cv, scoring="roc_auc", n_jobs=1)
    print(f"5-fold CV AUC {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
    row_def["CV_AUC"] = cv_scores.mean()

---
# 5 · Four variants of the same model

The tuned model is refitted three more ways, to answer whether the class
imbalance is what is holding the numbers down.

| Variant | What changes |
|---|---|
| `base` | the tuned model, unchanged |
| `smote` | `SMOTE` inside the pipeline, after the preprocessor, fitted only on training rows |
| `bagging` | `BalancedBaggingClassifier`: each member sees a bootstrap resample with the classes balanced |
| `calib` | isotonic recalibration of `base` by internal cross-validation |

**The search runs once, not four times.** The hyper-parameters come from the
`base` study and the variants change the sampling, not the model. Four searches
would cost four times as much and would confound the difference between variants
with the variation of the search itself.

**What to expect, so the result can contradict it.** Resampling does not create
information, so AUC rarely moves: SMOTE and bagging invent or duplicate cases,
they do not add signal. What they do change is the probability scale. A model
trained on a resampled world where half the people have the outcome reports the
risk of a population that does not exist, and the calibration intercept moves
away from zero. This is why the metrics table carries `calib_intercept` and
`calib_slope` alongside AUC: AUC is invariant to any monotone rescaling of the
probabilities and will not notice calibration breaking.

If the goal is a better-calibrated risk, `calib` is the variant built for it.
If the goal is discrimination, the honest prior is that none of the three moves
it, and the lever is elsewhere — in the outcome definition, not the sampler.

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedBaggingClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.calibration import CalibratedClassifierCV

EXPENSIVE_VARIANTS = False   # bagging and calib cost 10 and 5 API fits

def _final_estimator():
    """A fresh copy of the estimator the search selected."""
    return make_tabpfn(n_estimators=n_best)

VARIANTS = {}

def register(name, model_obj):
    """Fit, evaluate, and keep the probabilities for the figures."""
    model_obj.fit(Xtr, ytr)
    row, prob = mk.evaluate(model_obj, Xte, yte, f"{MODEL} ({name})")
    row["variant"] = name
    VARIANTS[name] = dict(model=model_obj, prob=prob, row=row)
    print(f"{name:<8} AUC {row['AUC_test']:.3f} "
          f"[{row['AUC_lo']:.3f}, {row['AUC_hi']:.3f}]  "
          f"Brier {row['Brier']:.4f}  "
          f"calibration intercept {row['calib_intercept']:+.2f} "
          f"slope {row['calib_slope']:.2f}")
    return row

# base: the model section 4 already fitted
row_base = row_tuned.copy()
row_base["variant"] = "base"
VARIANTS["base"] = dict(model=BEST_MODEL, prob=BEST_P, row=row_base)
print(f"{'base':<8} AUC {row_base['AUC_test']:.3f} "
      f"[{row_base['AUC_lo']:.3f}, {row_base['AUC_hi']:.3f}]  "
      f"Brier {row_base['Brier']:.4f}  "
      f"calibration intercept {row_base['calib_intercept']:+.2f} "
      f"slope {row_base['calib_slope']:.2f}")

# smote: resampling belongs after the encoder. Synthesising a case between two
# rows of raw questionnaire codes would invent categories that do not exist.
#
# The preprocessor's steps are spread into the chain rather than nested, because
# imblearn refuses a Pipeline as an intermediate step.
register("smote", ImbPipeline([*preproc().steps,
                               ("smote", SMOTE(random_state=mk.RS, k_neighbors=5)),
                               ("clf", _final_estimator())]))

if True and EXPENSIVE_VARIANTS :
    # The preprocessor stays outside the ensemble: imblearn refuses a Pipeline
    # as a bagged estimator, and resampling belongs on encoded columns anyway,
    # exactly where SMOTE sits.
    register("bagging", Pipeline([
        ("prep", preproc()),
        ("clf", BalancedBaggingClassifier(
            estimator=_final_estimator(), n_estimators=10,
            sampling_strategy="auto", replacement=False,
            random_state=mk.RS, n_jobs=1))]))

    register("calib", CalibratedClassifierCV(
        Pipeline([("prep", preproc()), ("clf", _final_estimator())]),
        method="isotonic", cv=5))

---
# 6 · Results

Measured on the test half, which nothing above was fitted on.

## 6.1 · Metrics

**What to look for.** `AUC_lo`/`AUC_hi` is a 1,000-draw stratified bootstrap
interval on the test AUC. Two operating points are reported: Youden's J, and the
screening point that holds sensitivity at about 90% — the one a screening
instrument would actually be set at, and the one where PPV shows what the
prevalence costs.

`calib_intercept` and `calib_slope` are where the variants separate. Zero and
one is perfect. An intercept below zero means the model reports a risk higher
than the one observed, which is what training on resampled data produces.

In [ ]:
row_def["variant"] = "untuned"
results = mk.metrics_frame([row_def] + [v["row"] for v in VARIANTS.values()])
display(results)

print("\ncalibration, one line per variant")
cal = pd.DataFrame([{"variant": k,
                     "AUC": v["row"]["AUC_test"],
                     "Brier": v["row"]["Brier"],
                     "intercept": v["row"]["calib_intercept"],
                     "slope": v["row"]["calib_slope"],
                     "mean predicted": v["row"]["mean_predicted"],
                     "observed": v["row"]["observed"]}
                    for k, v in VARIANTS.items()])
display(cal.round(4))

## 6.2 · Confusion matrices

The 2x2 at both operating points, for every variant. The percentage in each cell
is the share of its **true** row: of the people who have the outcome, how many
the model flags, and of the people who do not, how many it flags anyway.

**What to look for.** At Youden the matrix is balanced by construction and says
little on its own. The screening point is the one that matters: holding
sensitivity near 90% in a cohort where the outcome is uncommon means flagging a
large share of everyone, and the false-positive cell is the cost of the
instrument.

In [ ]:
for name, v in VARIANTS.items():
    for point, label in ((v["row"]["youden_threshold"], "Youden"),
                         (v["row"]["screen90_threshold"], "~90% sensitivity")):
        print(f"\n{name} · {label}")
        display(mk.confusion_frame(yte, v["prob"], point))

figs_cm = {}
for name, v in VARIANTS.items():
    figs_cm[name] = mk.plot_confusion(
        yte, v["prob"], v["row"]["screen90_threshold"],
        f"{OUTCOME} · {MODEL} · {name} · screening point")

## 6.3 · Permutation importance

**What this does.** Shuffles one predictor at a time in the test set and measures
how far AUC falls, then sums the drops per registry block. Restricted to the 20 strongest features with 3 repeats: every permutation is a billed API round trip.

**What to look for.** The block table is the null-result table of the decision
log: for hypertension, access to care contributed exactly 0.000 and sleep less
than that. A block that suddenly matters for a new outcome is a finding; a block
that matters *too much* is usually leakage, and the first thing to check is
whether a counter was rebuilt.

In [ ]:
imp, blocks_imp = mk.permutation_report(BEST_MODEL, Xte, yte, bundle, n_repeats=3, top=20)
display(imp.head(20))
display(blocks_imp)

## 6.4 · Figures

One ROC with all four variants, the calibration curve of `base`, and the top 20
predictors. Written at 300 dpi.

Four curves that lie on top of each other is the expected picture, and it is the
answer to the question that started this: resampling moves the probabilities,
not the ranking.

In [ ]:
fig_roc = mk.plot_roc({name: (yte, v["prob"]) for name, v in VARIANTS.items()},
                      f"{OUTCOME} · {MODEL} · four variants")
fig_cal = mk.plot_calibration(yte, BEST_P, f"{OUTCOME} · calibration, base")
fig_imp = mk.plot_importance(imp, f"{OUTCOME} · permutation importance")

---
# 7 · Export

One folder per variant, under `outputs/models/<outcome>/<family>_<variant>/`:
the metrics row, the importance tables, the best parameters, the figures and a
plain-text report carrying the build hashes.

**The zip carries a timestamp**, `<outcome>_<family>_<variant>_<YYYYMMDD-HHMM>.zip`,
so two runs of the same cell do not overwrite each other in the Downloads folder
and the comparison notebook can keep the most recent of each combination.

**Where it goes.** `PUSH_RESULTS = True` commits the folder to
[isasaade-23/pns2013-model-runs](https://github.com/isasaade-23/pns2013-model-runs),
which is private — these are unpublished results. It needs a GitHub token in
the Colab saved keys named `GITHUB_TOKEN`: a fine-grained token with
**Contents: read and write** on that repository, enabled for this notebook.
Without a token, or with `PUSH_RESULTS = False`, the run zips itself into
`outputs/models/` instead, next to the folder it just wrote. That zip only goes
through the browser when there is nowhere durable to keep it: with
`MOUNT_DRIVE = True` it lands in the shared folder and nothing is downloaded,
and without Drive it lives on the session disk, which dies with the session, so
a copy is downloaded as well.

In [ ]:
import datetime

PUSH_RESULTS = False       # True -> commit to the private run repository as well
STAMP = datetime.datetime.utcnow().strftime("%Y%m%d-%H%M")

for name, v in VARIANTS.items():
    tag = f"{MODEL}_{name}"
    is_base = name == "base"
    folder = mk.export(
        OUTDIR, OUTCOME, tag, bundle,
        results[results["variant"] == name],
        importance=imp if is_base else None,
        blocks=blocks_imp if is_base else None,
        best_params={**BEST_PARAMS, "variant": name},
        figures=([("roc", fig_roc), ("calibration", fig_cal),
                  ("importance", fig_imp)] if is_base else [])
                + [("confusion", figs_cm[name])])

    if PUSH_RESULTS:
        try:
            mk.push_results(folder)
            continue
        except Exception as e:
            print("not pushed:", e)

    z = mk.zip_folder(folder, os.path.join(OUTDIR, f"{OUTCOME}_{tag}_{STAMP}.zip"))
    if not os.path.isdir(DRIVE):
        try:
            from google.colab import files
            files.download(z)
        except Exception as e:
            print(e)

results.to_csv(os.path.join(OUTDIR, f"table_{OUTCOME}_{MODEL}_{STAMP}.csv"),
               index=False)
print(f"\n{len(VARIANTS)} variants exported, stamp {STAMP}")

## 7.1 · Re-push a run that is already on disk

**What this does.** Sends the folder the cell above wrote, without refitting
anything. Use it when the export did not push: a session that cloned before the
last commit and ran an older `pns_modelkit`, a missing token, a connection that
dropped. Set `RETRY_PUSH = True` and run this cell alone.

**What to look for.** The link it prints. If it says no token, add `GITHUB_TOKEN`
to the Colab saved keys and run it again — the results are on disk either way,
and nothing has to be recomputed.

In [ ]:
RETRY_PUSH = False

if RETRY_PUSH:
    import importlib
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "origin", "main"])
    subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"])
    import pns_modelkit as mk
    importlib.reload(mk)
    for name in VARIANTS:
        mk.push_results(os.path.join(OUTDIR, OUTCOME, f"{MODEL}_{name}"))